# 13 - File I/O


NumPy reads and writes arrays to disk in two main styles:

### Text files (human readable)

| Function | Purpose |
|---|---|
| `np.loadtxt` | read a numeric text file |
| `np.genfromtxt` | read text with missing values (`NaN`) |
| `np.savetxt` | write an array to text |

### Binary files (fast, compact)

| Function | Purpose |
|---|---|
| `np.save("f.npy", a)` | save one array |
| `np.load("f.npy")` | load it back |
| `np.savez("f.npz", a=a, b=b)` | save **several** arrays |
| `a.tofile` / `np.fromfile` | raw bytes (no shape metadata) |
| `np.memmap` | map a file lazily — huge arrays without loading into RAM |

> **`savetxt` gotchas:** write floats explicitly (avoid `%.18e` bloat) with `fmt=`, and set `delimiter=`,`header=`, `comments=` as needed.

In [1]:
import numpy as np
import os

a = np.arange(12).reshape(3, 4)
a

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

## 1. Text files

Write with `savetxt`, read back with `loadtxt`.

In [2]:
np.savetxt("matrix.txt", a, fmt="%d", delimiter=",", header="demo matrix", comments="# ")
print(open("matrix.txt").read())

# demo matrix
0,1,2,3
4,5,6,7
8,9,10,11



In [3]:
back = np.loadtxt("matrix.txt", delimiter=",")
back

array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.]])

## 2. Reading messy text with `genfromtxt`

Missing entries become `NaN` instead of crashing the read.

In [4]:
with open("messy.csv", "w") as fh:
    fh.write("1,2,3\n4,,6\n7,8,9\n")

np.genfromtxt("messy.csv", delimiter=",")

array([[ 1.,  2.,  3.],
       [ 4., nan,  6.],
       [ 7.,  8.,  9.]])

In [5]:
# Clean up the text files
os.remove("matrix.txt")
os.remove("messy.csv")
print("temp text files removed")

temp text files removed


## 3. Binary `.npy` files

`save`/`load` keep the exact dtype and shape — fast and lossless.

In [6]:
np.save("data.npy", a)
reloaded = np.load("data.npy")
print("same shape:", reloaded.shape == a.shape, " same dtype:", reloaded.dtype == a.dtype)

same shape: True  same dtype: True


## 4. `.npz` — several arrays in one file

`savez` stores multiple named arrays; access them by name.

In [7]:
b = np.array([100, 200, 300])
np.savez("bundle.npz", matrix=a, vector=b)

bundle = np.load("bundle.npz")
print("stored names:", bundle.files)
print("vector:", bundle["vector"])

stored names: ['matrix', 'vector']
vector: [100 200 300]


## 5. Raw bytes with `tofile` / `fromfile`

No metadata — you must know the dtype and shape yourself.

In [8]:
a.tofile("raw.bin")
raw = np.fromfile("raw.bin", dtype=int).reshape(3, 4)
raw

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11]])

## 6. Memory-mapped files (`memmap`)

For huge arrays, `np.memmap` reads only what you touch — no full load into RAM.

In [9]:
with open("lazy.bin", "wb") as fh:
    pass                        # Windows needs the file to exist before mapping
mm = np.memmap("lazy.bin", dtype="float64", mode="w+", shape=(1000, 1000))
mm[0, :] = 1.0
mm[999, 999] = 42.0
mm.flush()
print("written lazily:", mm[0, 0], mm[999, 999])
del mm   # release the file handle so it can be removed

written lazily: 1.0 42.0


In [ ]:
# Clean up the demo files (ignore failures if a file is still open elsewhere)
for f in ["data.npy", "bundle.npz", "raw.bin", "lazy.bin"]:
    try:
        if os.path.exists(f):
            os.remove(f)
    except OSError:
        pass
print("demo files cleaned up")

## Summary

- Text: `loadtxt` / `genfromtxt` / `savetxt` (human readable).
- Binary: `save` / `load` for one array, `savez` / `load` for many.
- Raw: `tofile` / `fromfile`; huge data: `np.memmap`.

Next: **14-performance** — memory layout and vectorisation tricks.